# Novo Cabrais (RS) e vizinhos: Censo 2010 x Censo 2022

Compara **Novo Cabrais** com os **4 municípios que fazem divisa com ele** (Cachoeira do Sul, Candelária, Cerro Branco e Paraíso do Sul), com o **Rio Grande do Sul** e o **Brasil** como referência.

Temas: população, idade, urbano/rural, cor ou raça, domicílios, saneamento, alfabetização, trabalho e renda, economia (PIB) e agropecuária.

Tudo vem da API do SIDRA (apisidra.ibge.gov.br), que funciona no Colab. Cada consulta traz os 5 municípios de uma vez. O download completo leva de 2 a 4 minutos.

Valores em reais de 2010 são corrigidos pelo IPCA, para comparar poder de compra. No fim, o notebook gera uma planilha, os gráficos em PNG (prontos para a matéria) e um resumo automático com os destaques.

In [ ]:
import re, time, os, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'font.size': 10})

MUNICIPIOS = {   # código IBGE: nome (Novo Cabrais e os 4 municípios que fazem divisa com ele)
    '4313391': 'Novo Cabrais',
    '4303004': 'Cachoeira do Sul',
    '4304200': 'Candelária',
    '4305132': 'Cerro Branco',
    '4314027': 'Paraíso do Sul',
}
FOCO_COD, FOCO = '4313391', 'Novo Cabrais'   # município em destaque
REFERENCIAS = {'RS': 'n3/43', 'Brasil': 'n1/1'}
MUNIS = list(MUNICIPIOS.values())
LOCAIS = MUNIS + list(REFERENCIAS)
NIVEIS = {'n6/' + ','.join(MUNICIPIOS): None, **{nivel: nome for nome, nivel in REFERENCIAS.items()}}
COL_LOCAL = ['Município (Código)', 'Unidade da Federação (Código)', 'Brasil (Código)']
API = 'https://apisidra.ibge.gov.br/values'
_cache = {}

def sidra(caminho, tentativas=4):
    """Baixa uma consulta do SIDRA e devolve um DataFrame com as colunas nomeadas pelo cabeçalho."""
    url = f'{API}/{caminho.strip("/")}'
    if url in _cache:
        return _cache[url]
    for i in range(tentativas):
        try:
            r = requests.get(url, timeout=180)
            if r.status_code == 400:
                raise ValueError(f'consulta inválida: {r.text[:200]}')
            r.raise_for_status()
            dados = r.json()
            df = pd.DataFrame(dados[1:]).rename(columns=dados[0])
            df['Valor'] = pd.to_numeric(df['Valor'], errors='coerce')   # '-' = zero, '..'/'...' = sem dado
            _cache[url] = df
            return df
        except ValueError:
            raise
        except Exception as e:
            print(f'  tentativa {i+1} falhou: {e}')
            time.sleep(2 ** (i + 1))
    raise RuntimeError(url)

def por_local(consulta):
    """A mesma consulta para os 5 municípios (numa só chamada), o RS e o Brasil, com a coluna 'local'."""
    partes = []
    for nivel, nome in NIVEIS.items():
        df = sidra(f'{consulta}/{nivel}').copy()
        col = next(c for c in COL_LOCAL if c in df.columns)
        df['local'] = df[col].map(MUNICIPIOS) if nome is None else nome
        partes.append(df)
    return pd.concat(partes, ignore_index=True)

def num(consulta):
    """Soma dos valores da consulta para cada local. '-' do IBGE (zero) vira 0."""
    d = por_local(consulta)
    return d['Valor'].fillna(0).groupby(d['local']).sum().reindex(LOCAIS)

def valor(consulta):
    """Um valor único (média, mediana, taxa) para cada local; NaN se o IBGE não publicou."""
    d = por_local(consulta)
    return d.groupby('local')['Valor'].first().reindex(LOCAIS)

def br(x, casas=1):
    """Número no formato brasileiro: 1.234,5"""
    if pd.isna(x) or np.isinf(x):
        return 's/d'
    return f'{x:,.{casas}f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

# Inflação pelo IPCA: julho (data dos Censos) e dezembro (produção agropecuária anual)
ipca = sidra('t/1737/v/2266/p/201007,202207,201012,202212/n1/1').set_index('Mês (Código)')['Valor']
INFLACAO = ipca['202207'] / ipca['201007']
INFLACAO_DEZ = ipca['202212'] / ipca['201012']
print(f'R$ 1,00 de jul/2010 = R$ {br(INFLACAO, 2)} de jul/2022')
print(f'R$ 1,00 de dez/2010 = R$ {br(INFLACAO_DEZ, 2)} de dez/2022')

os.makedirs('graficos', exist_ok=True)

## 1. Mapa da região

In [ ]:
# Malha municipal do RS (tbrugz/geodata-br). Também confere quem faz divisa com Novo Cabrais.
import geopandas as gpd
malha = gpd.read_file('https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-43-mun.json')
malha['id'] = malha['id'].astype(str)
cand = malha.loc[malha.id == FOCO_COD, 'geometry'].iloc[0]
divisa = malha[(malha.id != FOCO_COD) & malha.intersects(cand.buffer(0.002))]
print('Fazem divisa com Novo Cabrais:', ', '.join(sorted(divisa['name'])))

regiao = malha[malha.id.isin(MUNICIPIOS)]
x0, y0, x1, y1 = regiao.total_bounds
fig, ax = plt.subplots(figsize=(8, 8))
malha.plot(ax=ax, color='#f0f0f0', edgecolor='white', lw=.6)
regiao.plot(ax=ax, color='#c9d6e3', edgecolor='white', lw=1)
malha[malha.id == FOCO_COD].plot(ax=ax, color='#c0392b', edgecolor='white', lw=1)
for _, m in regiao.iterrows():
    p = m.geometry.representative_point()
    ax.annotate(MUNICIPIOS[m.id], (p.x, p.y), ha='center', fontsize=9,
                color='white' if m.id == FOCO_COD else '#222', weight='bold' if m.id == FOCO_COD else 'normal')
ax.set_xlim(x0 - .1, x1 + .1); ax.set_ylim(y0 - .1, y1 + .1)
ax.set_axis_off()
ax.set_title('Novo Cabrais e os municípios vizinhos', loc='left', weight='bold')
ax.text(0, -.02, 'Fonte: IBGE, malha municipal', transform=ax.transAxes, fontsize=8, color='#666')
plt.tight_layout(); plt.savefig('graficos/00_mapa_regiao.png', dpi=200, bbox_inches='tight'); plt.show()

## 2. População e idade

In [ ]:
def idades(ano):
    """População por idade simples (0 a 100+) e sexo, para cada local."""
    if ano == 2010:
        base = 't/1378/v/93/p/2010/c1/0/c455/0/c287/all'
        sexos = {'Total': 'c2/0', 'Homens': 'c2/4', 'Mulheres': 'c2/5'}
    else:
        base = 't/9514/v/93/p/2022/c286/113635/c287/all'
        sexos = {'Total': 'c2/6794', 'Homens': 'c2/4', 'Mulheres': 'c2/5'}
    out = []
    for sexo, cs in sexos.items():
        df = por_local(f'{base}/{cs}')
        col = next(c for c in df.columns if df[c].astype(str).eq('Menos de 1 ano').any())
        for local, cat, v in zip(df['local'], df[col], df['Valor'].fillna(0)):
            m = re.fullmatch(r'(\d+) anos?( ou mais)?', cat)
            if cat == 'Menos de 1 ano':
                out.append((local, sexo, 0, v))
            elif m and (not m.group(2) or m.group(1) == '100'):   # '70 anos ou mais' é grupo, não idade
                out.append((local, sexo, int(m.group(1)), v))
    t = pd.DataFrame(out, columns=['local', 'sexo', 'idade', 'pessoas'])
    return t.groupby(['local', 'sexo', 'idade'])['pessoas'].sum()

idade = {2010: idades(2010), 2022: idades(2022)}

def ind_idade(s):
    s = s.xs('Total', level='sexo')
    r = {}
    for local in LOCAIS:
        p = s.loc[local]
        tot = p.sum()
        jovens, ativos, idosos = p.loc[0:14].sum(), p.loc[15:64].sum(), p.loc[65:].sum()
        acum = p.cumsum() / tot
        r[local] = {
            'População (soma das idades)': tot,
            '% de 0 a 14 anos': 100 * jovens / tot,
            '% de 15 a 64 anos': 100 * ativos / tot,
            '% de 65 anos ou mais': 100 * idosos / tot,
            'Idade mediana (anos)': acum[acum >= .5].index[0],
            'Índice de envelhecimento (idosos 65+ por 100 jovens 0-14)': 100 * idosos / jovens,
            'Razão de dependência (0-14 e 65+ por 100 de 15-64)': 100 * (jovens + idosos) / ativos,
        }
    return pd.DataFrame(r).T

ii = {ano: ind_idade(idade[ano]) for ano in idade}
pd.concat({2010: ii[2010], 2022: ii[2022]}, axis=1).round(1)

In [ ]:
# Pirâmide etária de Novo Cabrais, 2010 x 2022 (grupos de 5 anos, % da população)
def piramide(local, arquivo=None):
    fig, ax = plt.subplots(figsize=(8, 6.5))
    for ano, cor, estilo in [(2022, '#c0392b', 'bar'), (2010, '#2c3e50', 'contorno')]:
        s = idade[ano].loc[local]
        tot = s.xs('Total').sum()
        g = lambda sexo: s.xs(sexo).groupby(lambda i: min(i // 5, 18)).sum() / tot * 100
        h, m = g('Homens'), g('Mulheres')
        y = np.arange(len(h))
        if estilo == 'bar':
            ax.barh(y, -h, color=cor, alpha=.6, label=f'{ano}')
            ax.barh(y, m, color=cor, alpha=.6)
        else:
            ax.barh(y, -h, fill=False, edgecolor=cor, lw=1.2, label=f'{ano} (contorno)')
            ax.barh(y, m, fill=False, edgecolor=cor, lw=1.2)
    rot = [f'{5*i}-{5*i+4}' for i in range(18)] + ['90+']
    ax.set_yticks(range(19)); ax.set_yticklabels(rot)
    ax.axvline(0, c='k', lw=.5)
    ax.set_xticks(ax.get_xticks()); ax.set_xticklabels([f'{abs(x):.0f}%' for x in ax.get_xticks()])
    ax.set_title(f'{local}: pirâmide etária (à esquerda homens, à direita mulheres)', loc='left', weight='bold')
    ax.text(0, -.08, 'Fonte: IBGE, Censos 2010 e 2022', transform=ax.transAxes, fontsize=8, color='#666')
    ax.legend(loc='upper right'); plt.tight_layout()
    if arquivo:
        plt.savefig(arquivo, dpi=200, bbox_inches='tight')
    plt.show()

piramide(FOCO, 'graficos/01_piramide_novo_cabrais.png')

## 3. Tabela geral de indicadores

In [ ]:
linhas = []

def add(tema, nome, v2010, v2022, obs=''):
    for local in LOCAIS:
        linhas.append({'tema': tema, 'indicador': nome, 'local': local,
                       '2010': v2010[local], '2022': v2022[local], 'obs': obs})

pct = lambda parte, todo: 100 * parte / todo

# --- População
P10 = num('t/1378/v/93/p/2010/c1/0/c2/0/c287/0/c455/0')
P22 = num('t/9514/v/93/p/2022/c2/6794/c287/100362/c286/113635')
add('População', 'População total', P10, P22)
add('População', '% de mulheres',
    pct(num('t/1378/v/93/p/2010/c1/0/c2/5/c287/0/c455/0'), P10),
    pct(num('t/9514/v/93/p/2022/c2/5/c287/100362/c286/113635'), P22))
add('População', '% morando na zona urbana',
    pct(num('t/1378/v/93/p/2010/c1/1/c2/0/c287/0/c455/0'), P10),
    pct(num('t/9923/v/93/p/2022/c1/1'), num('t/9923/v/93/p/2022/c1/6795')))
for col in ['% de 0 a 14 anos', '% de 15 a 64 anos', '% de 65 anos ou mais', 'Idade mediana (anos)',
            'Índice de envelhecimento (idosos 65+ por 100 jovens 0-14)',
            'Razão de dependência (0-14 e 65+ por 100 de 15-64)']:
    add('Idade', col, ii[2010][col], ii[2022][col])

# --- Cor ou raça
for nome, c in [('% branca', '2776'), ('% preta', '2777'), ('% parda', '2779'), ('% negra (preta + parda)', '2777,2779')]:
    add('Cor ou raça', nome,
        pct(num(f't/3175/v/93/p/2010/c86/{c}/c1/0/c2/0/c287/0'), P10),
        pct(num(f't/9605/v/93/p/2022/c86/{c}'), num('t/9605/v/93/p/2022/c86/95251')))

# --- Domicílios e saneamento
add('Domicílios', 'Domicílios particulares permanentes', num('t/3451/v/96/p/2010/c1/6795'), num('t/4712/v/381/p/2022'),
    '2022: somente os ocupados')
add('Domicílios', 'Média de moradores por domicílio', valor('t/3451/v/246/p/2010/c1/6795'), valor('t/4712/v/5930/p/2022'))
D10 = 't/3218/v/96/p/2010'
tot10 = num(f'{D10}/c61/0/c299/0/c67/0/c309/0')
add('Saneamento', '% de domicílios com água da rede geral',
    pct(num(f'{D10}/c61/92853/c299/0/c67/0/c309/0'), tot10),
    pct(num('t/6803/v/381/p/2022/c1821/72144'), num('t/6803/v/381/p/2022/c1821/72129')),
    '2022: ligados à rede e usando como forma principal')
add('Saneamento', '% com esgoto em rede geral ou fossa séptica',
    pct(num(f'{D10}/c61/0/c299/9678,9679/c67/0/c309/0'), tot10),
    pct(num('t/6805/v/381/p/2022/c11558/72110,72111,72112'), num('t/6805/v/381/p/2022/c11558/46292')),
    'Inclui fossa séptica (e fossa filtro em 2022)')
add('Saneamento', '% com lixo coletado',
    pct(num(f'{D10}/c61/0/c299/0/c67/2520/c309/0'), tot10),
    pct(num('t/6892/v/381/p/2022/c67/2520'), num('t/6892/v/381/p/2022/c67/10972')))

# --- Alfabetização 15+
pop15_10 = idade[2010].xs('Total', level='sexo').loc[(slice(None), slice(15, None))].groupby(level='local').sum()
alf15_10 = num('t/3213/v/2591/p/2010/c287/100362') - num('t/3213/v/2591/p/2010/c287/93084,93085')
A22 = 't/9542/v/950/p/2022/c2/6794/c86/95251/c287/100362'
add('Educação', 'Taxa de alfabetização, 15 anos ou mais (%)',
    pct(alf15_10, pop15_10), pct(num(f'{A22}/c59/1023'), num(f'{A22}/c59/93024')))
add('Educação', 'Pessoas de 15+ que não sabem ler e escrever',
    pop15_10 - alf15_10, num(f'{A22}/c59/1024'))
print('Parte 1 pronta')

In [ ]:
# --- Trabalho e renda (rendimento de todos os trabalhos, pessoas de 10+ ocupadas)
R10 = 't/3577/p/2010/c1/6795/c1568/120704/c521/14583/c58/95253'
R22 = 't/10291/p/2022/c58/95253'
def renda(ano, var, sexo='total', cor='95251'):
    s = {'total': '6794', 'h': '4', 'm': '5'}[sexo]
    if ano == 2010:
        v = {'ocup': '916', 'media': '1985', 'mediana': '1984'}[var]
        return (num if var == 'ocup' else valor)(f'{R10}/v/{v}/c2/{s}/c86/{cor}')
    v = {'ocup': '844', 'media': '846', 'mediana': '845'}[var]
    return (num if var == 'ocup' else valor)(f'{R22}/v/{v}/c2/{s}/c86/{cor}')

add('Trabalho e renda', 'Pessoas ocupadas (10+)', renda(2010, 'ocup'), renda(2022, 'ocup'),
    'NÃO comparável: 2010 conta todos os ocupados (inclusive sem remuneração), 2022 só os com rendimento')
add('Trabalho e renda', 'Renda média do trabalho (R$ de jul/2022)', renda(2010, 'media') * INFLACAO, renda(2022, 'media'),
    '2010 corrigido pelo IPCA')
add('Trabalho e renda', 'Renda mediana do trabalho (R$ de jul/2022)', renda(2010, 'mediana') * INFLACAO, renda(2022, 'mediana'),
    '2010 corrigido pelo IPCA')
add('Trabalho e renda', 'Renda média nominal (R$ da época)', renda(2010, 'media'), renda(2022, 'media'))
add('Trabalho e renda', 'Mulheres ganham, para cada R$ 100 dos homens',
    100 * renda(2010, 'media', 'm') / renda(2010, 'media', 'h'),
    100 * renda(2022, 'media', 'm') / renda(2022, 'media', 'h'))

def razao_racial(ano):
    """Renda média de pretos e pardos (ponderada) para cada R$ 100 da renda média dos brancos.
    Se o IBGE não publicou a média de um grupo num município pequeno, ele fica de fora da conta."""
    soma, peso = 0, 0
    for c in ('2777', '2779'):
        n, m = renda(ano, 'ocup', cor=c), renda(ano, 'media', cor=c)
        ok = m.notna()
        soma = soma + (m * n).where(ok, 0)
        peso = peso + n.where(ok, 0)
    return 100 * (soma / peso.replace(0, np.nan)) / renda(ano, 'media', cor='2776')
add('Trabalho e renda', 'Negros ganham, para cada R$ 100 dos brancos', razao_racial(2010), razao_racial(2022),
    'Em municípios pequenos há poucos negros ocupados; ver a linha seguinte antes de usar')
add('Trabalho e renda', 'Pessoas negras ocupadas (base da linha anterior)',
    renda(2010, 'ocup', cor='2777,2779'), renda(2022, 'ocup', cor='2777,2779'), 'NÃO comparável entre os anos (ver ocupadas)')

# --- Economia (PIB dos Municípios)
def pib(ano):
    return num(f't/5938/v/37/p/{ano}')
add('Economia', 'PIB (milhões de R$ de jul/2022)', pib(2010) / 1000 * INFLACAO, pib(2022) / 1000,
    '2010 corrigido pelo IPCA de julho; PIB 2022 sujeito a revisão')
add('Economia', 'PIB por habitante (R$ de jul/2022)', pib(2010) * 1000 / P10 * INFLACAO, pib(2022) * 1000 / P22,
    'População do Censo de cada ano')
add('Economia', '% da economia vinda da agropecuária', valor('t/5938/v/516/p/2010'), valor('t/5938/v/516/p/2021'),
    'Coluna 2022 = dado de 2021, último publicado por setor')
add('Economia', '% da economia vinda da indústria', valor('t/5938/v/520/p/2010'), valor('t/5938/v/520/p/2021'),
    'Coluna 2022 = dado de 2021, último publicado por setor')

tab = pd.DataFrame(linhas)
tab['variação'] = tab['2022'] - tab['2010']
tab['variação %'] = 100 * tab['variação'] / tab['2010'].abs()
print('Pronto:', tab['indicador'].nunique(), 'indicadores para', len(LOCAIS), 'locais')

## 4. Comparação entre os municípios

In [ ]:
ordem = list(dict.fromkeys(tab['indicador']))
def quadro(coluna):
    return tab.pivot_table(index='indicador', columns='local', values=coluna, sort=False).reindex(index=ordem, columns=LOCAIS)

v2010, v2022, var = quadro('2010'), quadro('2022'), quadro('variação')
print('VALORES EM 2022'); display(v2022.round(1))

In [ ]:
print('QUANTO MUDOU DE 2010 PARA 2022 (pontos ou unidades do indicador)'); display(var.round(1))

In [ ]:
# Ranking de avanços: só indicadores em que fica claro o que é "melhorar".
# 'pontos' = diferença em pontos percentuais; '%' = crescimento percentual;
# 'igualdade' = quanto a razão se aproximou de 100 (igualdade entre os grupos).
AVANCO = {
    '% de domicílios com água da rede geral': ('pontos', 1),
    '% com esgoto em rede geral ou fossa séptica': ('pontos', 1),
    '% com lixo coletado': ('pontos', 1),
    'Taxa de alfabetização, 15 anos ou mais (%)': ('pontos', 1),
    'Pessoas de 15+ que não sabem ler e escrever': ('%', -1),
    'Renda média do trabalho (R$ de jul/2022)': ('%', 1),
    'Renda mediana do trabalho (R$ de jul/2022)': ('%', 1),
    'PIB por habitante (R$ de jul/2022)': ('%', 1),
    'Mulheres ganham, para cada R$ 100 dos homens': ('igualdade', 1),
    'Negros ganham, para cada R$ 100 dos brancos': ('igualdade', 1),
}
def avanco(ind):
    modo, sentido = AVANCO[ind]
    a, b = v2010.loc[ind], v2022.loc[ind]
    if modo == 'pontos':
        return sentido * (b - a)
    if modo == '%':
        return sentido * 100 * (b / a - 1)
    return (a - 100).abs() - (b - 100).abs()

av = pd.DataFrame({ind: avanco(ind) for ind in AVANCO}).T
rank = av[MUNIS].rank(axis=1, ascending=False, method='min')
rank.loc['Posição média'] = rank.mean()
print('AVANÇO 2010 -> 2022 (positivo = melhorou; pontos, % ou aproximação da igualdade)')
display(av.round(1))
print('\nPOSIÇÃO DE CADA MUNICÍPIO NO AVANÇO (1 = quem mais avançou entre os 5)')
display(rank.astype(float).round(1))
print('\nPOSIÇÃO EM 2022 (1 = melhor nível em 2022 entre os 5)')
nivel = {}
for ind, (modo, sentido) in AVANCO.items():
    x = v2022.loc[ind, MUNIS]
    if modo == 'igualdade':
        nivel[ind] = -(x - 100).abs()
    elif ind != 'Pessoas de 15+ que não sabem ler e escrever':   # contagem depende do tamanho; a taxa já entra
        nivel[ind] = sentido * x
pos2022 = pd.DataFrame(nivel).T.rank(axis=1, ascending=False, method='min')
display(pos2022.astype(float).round(0))

## 5. Gráficos para a matéria

In [ ]:
COR_CAND, COR_VIZ, COR_2010, COR_REF = '#c0392b', '#1f5f99', '#b8b8b8', '#555555'

def halteres(ind, titulo, arquivo, casas=1, sufixo='', locais=MUNIS + ['RS']):
    """Um ponto para 2010 (cinza) e outro para 2022 (cor), ordenado pelo valor de 2022."""
    d = pd.DataFrame({'2010': v2010.loc[ind, locais], '2022': v2022.loc[ind, locais]}).dropna()
    d = d.sort_values('2022')
    y = np.arange(len(d))
    fig, ax = plt.subplots(figsize=(8, 0.42 * len(d) + 1.4))
    for i, (local, r) in enumerate(d.iterrows()):
        cor = COR_CAND if local == FOCO else COR_REF if local in REFERENCIAS else COR_VIZ
        ax.plot([r['2010'], r['2022']], [i, i], color=cor, lw=2, alpha=.35, zorder=1)
        ax.scatter(r['2010'], i, s=55, color='white', edgecolor=COR_2010, lw=2, zorder=2)
        ax.scatter(r['2022'], i, s=70, color=cor, edgecolor='white', lw=1.5, zorder=3)
        lado = 'left' if r['2022'] >= r['2010'] else 'right'
        ax.annotate(f" {br(r['2022'], casas)}{sufixo} ", (r['2022'], i), va='center', ha=lado, fontsize=9,
                    color='#222', xytext=(4 if lado == 'left' else -4, 0), textcoords='offset points')
    ax.set_yticks(y)
    ax.set_yticklabels(d.index)
    for t in ax.get_yticklabels():
        if t.get_text() == FOCO:
            t.set_color(COR_CAND); t.set_weight('bold')
    ax.grid(axis='x', color='#e6e6e6', lw=.8); ax.set_axisbelow(True)
    ax.spines['left'].set_visible(False); ax.tick_params(axis='y', length=0)
    ax.scatter([], [], s=55, color='white', edgecolor=COR_2010, lw=2, label='2010')
    ax.scatter([], [], s=70, color=COR_VIZ, label='2022')
    ax.legend(loc='lower right', frameon=False)
    ax.margins(x=.12)
    ax.set_title(titulo, loc='left', weight='bold')
    ax.text(0, -0.9 / (0.42 * len(d) + 1.4), 'Fonte: IBGE, Censos 2010 e 2022', transform=ax.transAxes,
            fontsize=8, color='#666')
    plt.tight_layout(); plt.savefig(f'graficos/{arquivo}.png', dpi=200, bbox_inches='tight'); plt.show()

halteres('% com esgoto em rede geral ou fossa séptica', 'Domicílios com esgoto em rede ou fossa séptica (%)', '02_esgoto', sufixo='%')
halteres('% de domicílios com água da rede geral', 'Domicílios com água da rede geral (%)', '03_agua', sufixo='%')
halteres('% com lixo coletado', 'Domicílios com lixo coletado (%)', '04_lixo', sufixo='%')
halteres('Taxa de alfabetização, 15 anos ou mais (%)', 'Taxa de alfabetização, 15 anos ou mais (%)', '05_alfabetizacao', sufixo='%')
halteres('Renda mediana do trabalho (R$ de jul/2022)', 'Renda mediana do trabalho (R$ de jul/2022, 2010 corrigido pelo IPCA)', '06_renda_mediana', casas=0)
halteres('PIB por habitante (R$ de jul/2022)', 'PIB por habitante (R$ de jul/2022, 2010 corrigido pelo IPCA)', '07_pib_per_capita', casas=0)
halteres('% de 65 anos ou mais', 'Moradores com 65 anos ou mais (%)', '08_idosos', sufixo='%')
halteres('% morando na zona urbana', 'Moradores da zona urbana (%)', '09_urbano', sufixo='%')

In [ ]:
# Crescimento da população, 2010 -> 2022 (%)
d = (100 * (P22 / P10 - 1)).reindex(MUNIS + ['RS', 'Brasil']).sort_values()
fig, ax = plt.subplots(figsize=(8, 0.42 * len(d) + 1.4))
cores = [COR_CAND if l == FOCO else COR_REF if l in REFERENCIAS else COR_VIZ for l in d.index]
ax.barh(d.index, d.values, color=cores, height=.65)
ax.axvline(0, color='#333', lw=.8)
for i, v in enumerate(d.values):
    ax.annotate(f' {br(v)}% ', (v, i), va='center', ha='left' if v >= 0 else 'right', fontsize=9)
for t in ax.get_yticklabels():
    if t.get_text() == FOCO:
        t.set_color(COR_CAND); t.set_weight('bold')
ax.spines['left'].set_visible(False); ax.tick_params(axis='y', length=0)
ax.margins(x=.15)
ax.set_title('Variação da população entre os Censos de 2010 e 2022 (%)', loc='left', weight='bold')
ax.text(0, -0.9 / (0.42 * len(d) + 1.4), 'Fonte: IBGE, Censos 2010 e 2022', transform=ax.transAxes, fontsize=8, color='#666')
plt.tight_layout(); plt.savefig('graficos/10_populacao.png', dpi=200, bbox_inches='tight'); plt.show()

## 6. Agropecuária

In [ ]:
agro = []
def add_agro(nome, ano_a, va, ano_b, vb, obs=''):
    for local in LOCAIS:
        agro.append({'indicador': nome, 'local': local, 'ano inicial': ano_a, 'valor inicial': va[local],
                     'ano final': ano_b, 'valor final': vb[local], 'obs': obs})

# Censo Agropecuário 2006 x 2017
C06 = 't/1109/p/2006/c218/0'
C17 = 't/6880/p/2017/c12517/113601/c12568/113197/c220/110085'
est06, est17 = num(f'{C06}/v/183/c12896/0'), num(f'{C17}/v/183/c829/46302')
fam06, fam17 = num(f'{C06}/v/183/c12896/119107'), num(f'{C17}/v/183/c829/46304')
area06, area17 = num(f'{C06}/v/184/c12896/0'), num(f'{C17}/v/184/c829/46302')
obs_fam = 'A definição de agricultura familiar mudou um pouco entre os Censos (Lei 11.326 x Decreto 9.064)'
add_agro('Estabelecimentos agropecuários', 2006, est06, 2017, est17)
add_agro('% de estabelecimentos da agricultura familiar', 2006, pct(fam06, est06), 2017, pct(fam17, est17), obs_fam)
add_agro('Área dos estabelecimentos (ha)', 2006, area06, 2017, area17)
add_agro('Área média por estabelecimento (ha)', 2006, area06 / est06, 2017, area17 / est17)

# Lavouras (PAM): área plantada e valor da produção, 2010 x 2022
pam = por_local('t/5457/v/8331,215/p/2010,2022/c782/all')
pam = pam.rename(columns={'Produto das lavouras temporárias e permanentes': 'cultura'})
def lavoura(cultura, var, ano):
    s = pam[pam['cultura'].str.startswith(cultura) & (pam['Variável (Código)'] == var) & (pam['Ano'] == ano)]
    return s['Valor'].fillna(0).groupby(s['local']).sum().reindex(LOCAIS).fillna(0)
add_agro('Área plantada, todas as lavouras (ha)', 2010, lavoura('Total', '8331', '2010'), 2022, lavoura('Total', '8331', '2022'))
add_agro('Valor da produção das lavouras (milhões de R$ de dez/2022)', 2010,
         lavoura('Total', '215', '2010') / 1000 * INFLACAO_DEZ, 2022, lavoura('Total', '215', '2022') / 1000,
         '2010 corrigido pelo IPCA de dezembro')
for nome, cultura in [('fumo', 'Fumo'), ('soja', 'Soja'), ('milho', 'Milho'), ('arroz', 'Arroz')]:
    add_agro(f'Área plantada de {nome} (ha)', 2010, lavoura(cultura, '8331', '2010'), 2022, lavoura(cultura, '8331', '2022'))

# Rebanho bovino e leite (PPM)
reb = por_local('t/3939/v/105/p/2010,2022/c79/all')
bov = reb[reb['Tipo de rebanho'].str.startswith('Bovino')]
bov = bov.pivot_table(index='local', columns='Ano', values='Valor', aggfunc='sum').reindex(LOCAIS)
add_agro('Rebanho bovino (cabeças)', 2010, bov['2010'], 2022, bov['2022'])
ani = por_local('t/74/v/106/p/2010,2022/c80/all')
leite = ani[ani['Tipo de produto de origem animal'].str.startswith('Leite')]
leite = leite.pivot_table(index='local', columns='Ano', values='Valor', aggfunc='sum').reindex(LOCAIS)
add_agro('Produção de leite (mil litros)', 2010, leite['2010'], 2022, leite['2022'])

agro = pd.DataFrame(agro)
agro['variação'] = agro['valor final'] - agro['valor inicial']
agro['variação %'] = 100 * agro['variação'] / agro['valor inicial'].replace(0, np.nan)
ordem_agro = list(dict.fromkeys(agro['indicador']))
agro_final = agro.pivot_table(index='indicador', columns='local', values='valor final', sort=False).reindex(index=ordem_agro, columns=LOCAIS)
agro_var = agro.pivot_table(index='indicador', columns='local', values='variação %', sort=False).reindex(index=ordem_agro, columns=LOCAIS)
print('AGRO: VALOR MAIS RECENTE (2017 no Censo Agro, 2022 nas lavouras e rebanhos)'); display(agro_final.round(1))
print('\nAGRO: VARIAÇÃO %'); display(agro_var.round(1))

In [ ]:
# Área plantada por cultura em 2022 (hectares), município a município
cult = agro[agro.indicador.str.startswith('Área plantada de') & agro.local.isin(MUNIS)]
cult = cult.pivot_table(index='local', columns='indicador', values='valor final').reindex(MUNIS)
cult.columns = [c.replace('Área plantada de ', '').replace(' (ha)', '') for c in cult.columns]
cult = cult[['fumo', 'soja', 'milho', 'arroz']].sort_values('fumo')
ax = cult.plot.barh(figsize=(9, 5.5), color=['#8c6d31', '#1f5f99', '#e0a526', '#7aa874'], width=.8, edgecolor='white')
ax.set_xlabel('hectares'); ax.grid(axis='x', color='#e6e6e6'); ax.set_axisbelow(True)
ax.set_ylabel('')
ax.set_title('Área plantada das principais lavouras em 2022 (hectares)', loc='left', weight='bold')
ax.text(0, -.12, 'Fonte: IBGE, Produção Agrícola Municipal', transform=ax.transAxes, fontsize=8, color='#666')
ax.legend(frameon=False)
plt.tight_layout(); plt.savefig('graficos/11_lavouras_2022.png', dpi=200, bbox_inches='tight'); plt.show()

## 7. Resumo automático dos destaques

In [ ]:
# Uma frase por indicador: valor de Novo Cabrais, quem mais subiu e quem mais caiu entre os 5, e o RS.
def frase(nome, a, b, ano_a, ano_b, var_, casas=1):
    v = var_[MUNIS].replace([np.inf, -np.inf], np.nan).dropna()
    sobe, desce = v.idxmax(), v.idxmin()
    return (f'- {nome}: {FOCO} {br(a[FOCO], casas)} ({ano_a}) -> {br(b[FOCO], casas)} ({ano_b}). '
            f'Maior alta entre os 5: {sobe} ({br(v[sobe], casas)}); menor: {desce} ({br(v[desce], casas)}). '
            f'RS: {br(a["RS"], casas)} -> {br(b["RS"], casas)}.')

texto = ['CENSOS 2010 x 2022']
for ind in ordem:
    usa_pct = ind in ('População total', 'Domicílios particulares permanentes', 'Pessoas de 15+ que não sabem ler e escrever') \
              or 'R$' in ind
    variacao = 100 * (v2022.loc[ind] / v2010.loc[ind] - 1) if usa_pct else var.loc[ind]
    texto.append(frase(ind + (' [variação em %]' if usa_pct else ' [variação em pontos]'),
                       v2010.loc[ind], v2022.loc[ind], 2010, 2022, variacao))
texto.append('\nAGROPECUÁRIA')
for ind in ordem_agro:
    d = agro[agro.indicador == ind].set_index('local')
    texto.append(frase(ind + ' [variação em %]', d['valor inicial'], d['valor final'],
                       d['ano inicial'].iloc[0], d['ano final'].iloc[0], d['variação %']))
texto.append('\nPOSIÇÃO MÉDIA NO RANKING DE AVANÇOS (1 = quem mais avançou)')
for local, p in rank.loc['Posição média'].sort_values().items():
    texto.append(f'- {local}: {br(p)}')
texto = '\n'.join(texto)
print(texto)

## 8. Salvar tudo

In [ ]:
with pd.ExcelWriter('novo_cabrais_vizinhos_2010_2022.xlsx') as xls:
    v2022.round(2).to_excel(xls, sheet_name='Valores 2022')
    v2010.round(2).to_excel(xls, sheet_name='Valores 2010')
    var.round(2).to_excel(xls, sheet_name='Variação')
    av.round(2).to_excel(xls, sheet_name='Avanço')
    rank.round(1).to_excel(xls, sheet_name='Ranking de avanços')
    pos2022.to_excel(xls, sheet_name='Posição em 2022')
    agro_final.round(2).to_excel(xls, sheet_name='Agro valor final')
    agro.round(2).to_excel(xls, sheet_name='Agro dados longos', index=False)
    tab.round(2).to_excel(xls, sheet_name='Dados longos', index=False)
with open('resumo_destaques.txt', 'w', encoding='utf-8') as f:
    f.write(texto)
with zipfile.ZipFile('novo_cabrais_vizinhos_2010_2022.zip', 'w') as z:
    for arq in ['novo_cabrais_vizinhos_2010_2022.xlsx', 'resumo_destaques.txt'] + \
               [f'graficos/{g}' for g in sorted(os.listdir('graficos'))]:
        z.write(arq)
print('Salvo: novo_cabrais_vizinhos_2010_2022.zip (planilha, resumo e gráficos)')
try:
    from google.colab import files
    files.download('novo_cabrais_vizinhos_2010_2022.zip')
except ImportError:
    pass